# Casefile — build and evaluate the RAG pipeline
A fictional investigation grounded in an authentic archaeological catalogue. This notebook builds the same persisted store used by FastAPI. Run top to bottom from the project environment.

## 2.1 Load and inspect
Ten synthetic evidence files and two authentic page extracts. Authentic PDF: Teeter (2003), printed pp.122–123 / PDF pp.146–147. Original corpus is kept outside the public repository; preparation instructions are in README. No answer keys are indexed.

In [ ]:
import sys, os, json
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from backend.app.services.corpus import documents, chunk_documents
from backend.app.core.config import DATA
corpus = documents()
assert len(corpus) == 12, "Prepare the reference extracts before running"
for doc in corpus:
    print(doc['id'], doc['kind'], len(doc['text']), 'characters', 'stage', doc['stage'])
print('All text files loaded. Two PDF page extracts; no OCR used. Specialist glyphs elsewhere are not part of this case.')

## 2.2 Chunking strategy
Start with 170-word windows and 35-word overlap. The case records are short; this preserves useful local context without filling an Intel CPU model's prompt with the whole corpus. Overlap reduces broken sentence context. Page boundaries and source metadata are preserved. Fixed word windows are a baseline, not an optimality claim; tables and timestamps require careful inspection.

In [ ]:
chunks = chunk_documents(corpus)
print('Chunks:', len(chunks))
print(chunks[0])

## 2.3 Embeddings and vector store
Chroma's ONNX all-MiniLM-L6-v2 encoder produces 384-dimensional semantic embeddings on CPU. This avoids installing a large language model for embedding. Chroma persists the vectors and source metadata. First run downloads the embedding model to the project cache.

In [ ]:
from scripts.ingest import build
config = build()
config

## 2.4 Retrieval and prompting
Retrieve top five passages with a semantic score plus a small lexical overlap adjustment. Crucially, filter by unlocked stage inside Chroma. Authentic scholarship is separate from fictional observations. Use a fixed prompt asking for evidence-based statements, valid citations and uncertainty.

In [ ]:
from backend.app.services.retrieval import Retriever
from backend.app.services.generation import SYSTEM, Generator
retriever = Retriever()
print(SYSTEM)
for hit in retriever.retrieve('How do heart scarabs differ from funerary scarabs?', stage=1):
    print(hit['chunk_id'], hit['score'], hit['text'][:160])
assert all(h['stage'] == 1 for h in retriever.retrieve('Nadia admission and final inspection', stage=1))
questions = json.loads((ROOT/"evaluation/questions.json").read_text())
for q in questions:
    print(q["id"], [h["chunk_id"] for h in retriever.retrieve(q["question"], stage=2)])


## 2.5 Vision component
Core Track. No YOLO/CV model is used. The decorative scarab drawing is an illustration, not evidence or image recognition.

## 2.6 Evaluation
Eighteen authored questions cover facts, source comparisons, authentic references, and missing information. Expected-source coverage measures retrieval only. The answer/grounding columns require human review; they must not be confused with measured correctness.

This section displays the saved, reviewed live run. To generate new outputs, run scripts/evaluate.py --generate and review the new answers. If no result exists, the cell runs live evaluation and leaves judgments pending.

In [ ]:
# Reuse the recorded live run for reproducible review; run the evaluator explicitly to regenerate.
from scripts.evaluate import evaluate
from IPython.display import display, Markdown
import pandas as pd
results_path = ROOT / "evaluation/results.json"
if results_path.exists():
    results = json.loads(results_path.read_text())
else:
    results = evaluate(generate=True)
print("Generation requested:", results["generation_requested"])
display(pd.DataFrame(results["rows"])[["question","retrieved_sources","answer","correct_or_not"]])
display(Markdown((ROOT/"evaluation/RESULTS.md").read_text()))


### Observed and anticipated failure cases
Inspect the measured table for missed expected sources. A short top-k list can omit one side of a comparison. Unknown questions still retrieve nearby material, so the generator must not assume relevance means answerability. Citation IDs are validated against retrieved IDs, but this does not automatically verify that the passage supports each claim. The UI exposes sources for checking. Specialist glyph extraction is excluded from clue interpretation. A mandatory stage filter prevents unreleased evidence entering the prompt.

After reviewing the actual answers, update `evaluation/RESULTS.md` with supported judgments and concrete failures; do not invent correctness percentages.

## 2.7 Export and backend handoff
The store and configuration are already on disk. FastAPI loads them at startup; it does not re-embed on requests.

In [ ]:
assert (DATA/'vector_store/chroma.sqlite3').exists()
print((DATA/'vector_store/config.json').read_text())
print('Backend entry point: uvicorn backend.app.main:app --host 127.0.0.1 --port 8000')